In [1]:
%pip install opencv-python-headless -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import cv2
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Pretty print
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras version: {keras.__version__}")
print(f"✅ GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

Note: you may need to restart the kernel to use updated packages.
✅ TensorFlow version: 2.20.0
✅ Keras version: 3.13.2
✅ GPU available: False


In [2]:
# ─────────────────────────────────────────────
#  Configuration — edit paths here if needed
# ─────────────────────────────────────────────

TRAIN_PATH = '../input/dogs-cats-images/dog vs cat/dataset/training_set'
TEST_PATH  = '../input/dogs-cats-images/dog vs cat/dataset/test_set'

IMAGE_SIZE  = 128        # Resize images to this square size
BATCH_SIZE  = 32
EPOCHS      = 20
SEED        = 42
CLASS_NAMES = ['cats', 'dogs']  # update if your folder names differ

# Style
sns.set_style('whitegrid')
PALETTE = ['#764ba2', '#667eea']

print("✅ Configuration loaded")

✅ Configuration loaded


In [3]:
# ── Training generator with augmentation ──
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# ── Validation/test generator (no augmentation) ──
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # keep order for confusion matrix
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"\n📂 Classes: {class_names}")
print(f"🖼️  Training samples : {train_generator.samples}")
print(f"🖼️  Test samples     : {test_generator.samples}")


FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: '../input/dogs-cats-images/dog vs cat/dataset/training_set'

In [ ]:
# ── Count images per class ──
def count_images(generator):
    counts = {}
    for cls, idx in generator.class_indices.items():
        counts[cls] = np.sum(generator.classes == idx)
    return counts

train_counts = count_images(train_generator)
test_counts  = count_images(test_generator)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 Class Distribution', fontsize=16, fontweight='bold', y=1.02)

for ax, counts, title in zip(axes,
                              [train_counts, test_counts],
                              ['Training Set', 'Test Set']):
    bars = ax.bar(counts.keys(), counts.values(),
                  color=PALETTE, edgecolor='black', linewidth=0.8)
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, max(counts.values()) * 1.15)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# ── Display a 4×4 grid of sample images ──
def show_sample_grid(generator, title='Sample Images', n=16):
    images, labels = next(generator)
    class_labels   = list(generator.class_indices.keys())
    indices = np.random.choice(len(images), min(n, len(images)), replace=False)

    cols = 4
    rows = (len(indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    for ax, i in zip(axes.flat, indices):
        ax.imshow(images[i])
        label = class_labels[np.argmax(labels[i])]
        emoji = '🐶' if label == 'dogs' else '😺'
        ax.set_title(f'{emoji} {label}', fontsize=10, fontweight='bold')
        ax.axis('off')
    for ax in axes.flat[len(indices):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_grid(train_generator, title='Training Set — Sample Images (augmented)')

In [ ]:
# ── Show augmentation effects on a single image ──
sample_path = next(
    Path(TRAIN_PATH).rglob('*.jpg'),
    next(Path(TRAIN_PATH).rglob('*.png'), None)
)

if sample_path:
    raw = img_to_array(load_img(sample_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))) / 255.
    raw_batch = raw[np.newaxis, ...]

    # ✅ No rescale here — image is already in [0,1]
    aug_gen_viz = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        fill_mode='nearest'
    ).flow(raw_batch, batch_size=1)

    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    fig.suptitle('🔄 Data Augmentation — 9 variations of the same image',
                 fontsize=15, fontweight='bold')

    axes[0, 0].imshow(raw)
    axes[0, 0].set_title('Original', fontweight='bold')
    axes[0, 0].axis('off')

    for ax in list(axes.flat)[1:]:
        aug_img = next(aug_gen_viz)[0]
        ax.imshow(aug_img)
        ax.set_title('Augmented', color='#764ba2')
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Could not find a sample image — check TRAIN_PATH.')

In [ ]:
# ── Pixel intensity distribution per class ──
def sample_pixel_distributions(generator, n_batches=3):
    class_pixels = {cls: [] for cls in generator.class_indices.keys()}
    for _ in range(n_batches):
        imgs, lbls = next(generator)
        for img, lbl in zip(imgs, lbls):
            cls = list(generator.class_indices.keys())[np.argmax(lbl)]
            class_pixels[cls].append(img.mean(axis=(0, 1)))   # mean RGB per image
    return {k: np.array(v) for k, v in class_pixels.items()}

pixel_data = sample_pixel_distributions(train_generator)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['#e74c3c', '#2ecc71', '#3498db']

for ch, (ax, cname, ccolor) in enumerate(zip(axes, channel_names, channel_colors)):
    for cls, color in zip(pixel_data.keys(), PALETTE):
        ax.hist(pixel_data[cls][:, ch], bins=30, alpha=0.65, color=color, label=cls)
    ax.set_title(f'{cname} Channel Distribution', fontweight='bold')
    ax.set_xlabel('Mean Pixel Value (0-1)')
    ax.legend()
    sns.despine(ax=ax)

fig.suptitle('📈 Pixel Intensity Distribution by Class & Channel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def build_cnn(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), num_classes=2):
    """Custom CNN with 4 convolutional blocks."""
    inputs = keras.Input(shape=input_shape, name='input')

    # ── Block 1 ──
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1_1')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, name='pool1')(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 2 ──
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, name='pool2')(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 3 ──
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, name='pool3')(x)
    x = layers.Dropout(0.3)(x)

    # ── Block 4 ──
    x = layers.Conv2D(256, 3, padding='same', activation='relu', name='conv4_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, name='pool4')(x)
    x = layers.Dropout(0.3)(x)

    # ── Classification head ──
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs, outputs, name='DogsCats_CNN')

model = build_cnn(num_classes=num_classes)
model.summary()

In [ ]:
# ── Visual architecture diagram ──
try:
    keras.utils.plot_model(
        model,
        to_file='model_architecture.png',
        show_shapes=True,
        show_layer_names=True,
        dpi=96,
        rankdir='TB'
    )
    from IPython.display import Image as IPImage
    display(IPImage('model_architecture.png'))
except Exception as e:
    print(f'📌 Could not render graph (graphviz optional): {e}')
    print('   Model summary printed above instead.')

In [ ]:
# ── Bar chart: parameters per layer type ──
from collections import defaultdict

type_params = defaultdict(int)
for layer in model.layers:
    ltype = layer.__class__.__name__
    type_params[ltype] += layer.count_params()

type_params = {k: v for k, v in type_params.items() if v > 0}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(type_params.keys()), list(type_params.values()),
               color='#764ba2', edgecolor='black', linewidth=0.6)
for bar in bars:
    w = bar.get_width()
    ax.text(w + max(type_params.values()) * 0.01, bar.get_y() + bar.get_height()/2,
            f'{w:,}', va='center', fontsize=10)
ax.set_xlabel('Total Parameters')
ax.set_title('🧩 Parameter Count by Layer Type', fontsize=14, fontweight='bold')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

total = model.count_params()
print(f"\n🔢 Total parameters : {total:,}")
print(f"   Trainable        : {sum(np.prod(w.shape) for w in model.trainable_weights):,}")

In [ ]:
# ── Build a model that outputs selected intermediate activations ──
target_conv_layers = ['conv1_1', 'conv2_1', 'conv3_1', 'conv4_1']

activation_model = keras.Model(
    inputs=model.input,
    outputs=[model.get_layer(name).output for name in target_conv_layers]
)

# ── Grab a sample test image ──
sample_img_batch, sample_labels = next(test_generator)
sample_img = sample_img_batch[0:1]   # shape (1, H, W, 3)

activations = activation_model.predict(sample_img, verbose=0)

print(f"\n✅ Extracted activations from {len(activations)} layers")
for name, act in zip(target_conv_layers, activations):
    print(f"   {name:12s}  →  shape {act.shape}")

In [ ]:
# ── Plot the first 8 feature maps for each chosen layer ──
n_filters_show = 8

fig = plt.figure(figsize=(18, len(target_conv_layers) * 3.5))
gs  = gridspec.GridSpec(len(target_conv_layers), n_filters_show + 1,
                        wspace=0.05, hspace=0.4)

for row, (layer_name, act) in enumerate(zip(target_conv_layers, activations)):
    # Column 0: original image
    ax0 = fig.add_subplot(gs[row, 0])
    ax0.imshow(sample_img[0])
    ax0.set_title('Input', fontsize=9)
    ax0.set_ylabel(layer_name, fontsize=10, fontweight='bold', rotation=0,
                   labelpad=55, va='center')
    ax0.axis('off')

    for col in range(n_filters_show):
        ax = fig.add_subplot(gs[row, col + 1])
        fmap = act[0, :, :, col] if col < act.shape[-1] else np.zeros(act.shape[1:3])
        ax.imshow(fmap, cmap='viridis')
        ax.set_title(f'f{col}', fontsize=8)
        ax.axis('off')

fig.suptitle('🔍 CNN Feature Maps per Convolutional Layer', fontsize=16, fontweight='bold', y=1.01)
plt.show()

In [ ]:
# ── Visualize the learned weight kernels of conv1_1 ──
conv1_weights = model.get_layer('conv1_1').get_weights()[0]  # shape (3,3,3,32)
n_show = min(32, conv1_weights.shape[-1])
cols = 8
rows = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6))
fig.suptitle('🎨 Learned Convolutional Filters — conv1_1 (3×3)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < n_show:
        kernel = conv1_weights[:, :, :, i]
        # Normalize to [0,1] for display
        kernel = (kernel - kernel.min()) / (kernel.max() - kernel.min() + 1e-8)
        ax.imshow(kernel)
        ax.set_title(f'#{i}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Grad-CAM: highlight which regions the model focuses on ──
import tensorflow as tf

def grad_cam(model, img_array, layer_name='conv4_1'):
    """Compute Grad-CAM heatmap for a given image."""
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        pred_class = tf.argmax(predictions[0])
        loss = predictions[:, pred_class]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap, int(pred_class.numpy()), float(tf.reduce_max(predictions[0]).numpy())

def overlay_grad_cam(img, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_color   = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.
    return np.clip(img * (1 - alpha) + heatmap_color * alpha, 0, 1)

# Show Grad-CAM for 6 test images
imgs, lbls = next(test_generator)
n_show = 6

fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
fig.suptitle('🌡️  Grad-CAM — Model Attention Heatmaps', fontsize=15, fontweight='bold')

for i in range(n_show):
    img = imgs[i]
    heatmap, pred_idx, confidence = grad_cam(model, img[np.newaxis, ...])
    overlay = overlay_grad_cam(img, heatmap)
    true_cls = class_names[np.argmax(lbls[i])]
    pred_cls = class_names[pred_idx]
    color = 'green' if true_cls == pred_cls else 'red'

    axes[i, 0].imshow(img);       axes[i, 0].set_title('Original');     axes[i, 0].axis('off')
    axes[i, 1].imshow(heatmap, cmap='jet'); axes[i, 1].set_title('Heatmap'); axes[i, 1].axis('off')
    axes[i, 2].imshow(overlay);   axes[i, 2].axis('off')
    axes[i, 2].set_title(f'Pred: {pred_cls} ({confidence:.0%})\nTrue: {true_cls}',
                         color=color, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Learning rate schedule ──
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=500,
    decay_rate=0.9,
    staircase=False
)

# ── Compile ──
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ──
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("✅ Model compiled and callbacks ready")

In [ ]:
# ── Train ──
history = model.fit(
    train_generator,
    steps_per_epoch  = train_generator.samples // BATCH_SIZE,
    validation_data  = test_generator,
    validation_steps = test_generator.samples  // BATCH_SIZE,
    epochs           = EPOCHS,
    callbacks        = callbacks
)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('📉 Training History', fontsize=16, fontweight='bold')

    metrics = [('accuracy', 'Accuracy'), ('loss', 'Loss')]
    for ax, (metric, label) in zip(axes, metrics):
        train_vals = history.history[metric]
        val_vals   = history.history[f'val_{metric}']
        epochs     = range(1, len(train_vals) + 1)

        ax.plot(epochs, train_vals, 'o-', color='#667eea', linewidth=2, label=f'Train {label}')
        ax.plot(epochs, val_vals,   's--', color='#764ba2', linewidth=2, label=f'Val {label}')
        best_epoch = (np.argmax if metric == 'accuracy' else np.argmin)(val_vals)
        ax.axvline(best_epoch + 1, color='red', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch+1})')
        ax.set_title(label, fontsize=14, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(label)
        ax.legend()
        sns.despine(ax=ax)

    plt.tight_layout()
    plt.show()

plot_history(history)

In [ ]:
# ── Test-set evaluation ──
test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f"\n{'='*40}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"{'='*40}")

In [ ]:
# ── Predictions & confusion matrix ──
test_generator.reset()
y_prob = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🔢 Confusion Matrix', fontsize=16, fontweight='bold')

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2%'],
    ['Counts', 'Normalised']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Purples',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()

print("\n📋 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ── Visualise misclassified images ──
test_generator.reset()
all_imgs, all_true, all_pred, all_conf = [], [], [], []

for imgs, lbls in test_generator:
    probs = model.predict(imgs, verbose=0)
    preds = np.argmax(probs, axis=1)
    trues = np.argmax(lbls, axis=1)
    confs = np.max(probs, axis=1)
    all_imgs.extend(imgs)
    all_true.extend(trues)
    all_pred.extend(preds)
    all_conf.extend(confs)
    if len(all_imgs) >= test_generator.samples:
        break

wrong_idx = [i for i, (t, p) in enumerate(zip(all_true, all_pred)) if t != p]
print(f"❌ Misclassified: {len(wrong_idx)} / {len(all_true)} ({len(wrong_idx)/len(all_true):.1%})")

n_show = min(12, len(wrong_idx))
selected = np.random.choice(wrong_idx, n_show, replace=False)

cols = 4; rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
fig.suptitle('❌ Misclassified Examples', fontsize=14, fontweight='bold')

for ax, i in zip(axes.flat, selected):
    ax.imshow(all_imgs[i])
    ax.set_title(f'True: {class_names[all_true[i]]}\nPred: {class_names[all_pred[i]]} ({all_conf[i]:.0%})',
                 color='red', fontsize=9)
    ax.axis('off')
for ax in axes.flat[n_show:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def predict_single_image(img_path_or_array, model, class_names, image_size=IMAGE_SIZE):
    """
    Predict class for a single image.

    Parameters
    ----------
    img_path_or_array : str | np.ndarray
        Path to an image file, or a pre-loaded numpy array (H, W, 3) in [0,1].
    """
    if isinstance(img_path_or_array, (str, Path)):
        img = load_img(img_path_or_array, target_size=(image_size, image_size))
        img_array = img_to_array(img) / 255.
        display_img = img_array
    else:
        img_array = img_path_or_array
        display_img = img_array

    pred = model.predict(img_array[np.newaxis, ...], verbose=0)[0]
    pred_idx  = np.argmax(pred)
    pred_cls  = class_names[pred_idx]
    confidence = pred[pred_idx]

    # ── Plot ──
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 4),
                                          gridspec_kw={'width_ratios': [1, 1.5]})

    ax_img.imshow(display_img)
    emoji = '🐶' if pred_cls == 'dogs' else '😺'
    ax_img.set_title(f'{emoji}  Prediction: {pred_cls.upper()}\n({confidence:.1%} confidence)',
                     fontsize=14, fontweight='bold',
                     color='#2ecc71' if confidence > 0.75 else '#e74c3c')
    ax_img.axis('off')

    # Confidence bar chart
    colors = ['#667eea' if i == pred_idx else '#cccccc' for i in range(len(class_names))]
    bars = ax_bar.barh(class_names, pred, color=colors, edgecolor='black', linewidth=0.7)
    for bar, val in zip(bars, pred):
        ax_bar.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.1%}', va='center', fontweight='bold')
    ax_bar.set_xlim(0, 1.1)
    ax_bar.set_xlabel('Confidence')
    ax_bar.set_title('Class Probabilities', fontweight='bold')
    sns.despine(ax=ax_bar)

    plt.suptitle('🔬 Single-Image Inference', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    return pred_cls, float(confidence)

print("✅ predict_single_image() defined")

In [ ]:
# ── Run inference on random test-set images ──
test_generator.reset()
sample_batch, sample_labels = next(test_generator)

for i in np.random.choice(len(sample_batch), 4, replace=False):
    predict_single_image(sample_batch[i], model, class_names)

In [ ]:
# ── Run inference on a CUSTOM image ──
# Edit the path below to your own image:
CUSTOM_IMAGE_PATH = "/kaggle/input/dogs-cats-images/dataset/test_set/cats/cat.4024.jpg"   # e.g. '/path/to/my_dog.jpg'

if CUSTOM_IMAGE_PATH and Path(CUSTOM_IMAGE_PATH).exists():
    label, conf = predict_single_image(CUSTOM_IMAGE_PATH, model, class_names)
    print(f"\n🏷️  Predicted: {label}  (confidence {conf:.1%})")
else:
    print("ℹ️  Set CUSTOM_IMAGE_PATH to run inference on your own image.")